In [1]:
!pip install ipywidgets

1.IMPORTS

In [2]:
import cv2
import os
import numpy as np
import tensorflow as tf
from tensorflow.keras.preprocessing.image import img_to_array
from tensorflow.keras.applications.efficientnet import preprocess_input
from PIL import Image
import matplotlib.pyplot as plt
from IPython.display import display, HTML, clear_output
import ipywidgets as widgets
from io import BytesIO

print("✅ All libraries imported successfully!")


d:\anaconda\Lib\site-packages\keras\src\export\tf2onnx_lib.py:8: FutureWarning: In the future `np.object` will be defined as the corresponding NumPy scalar.
  if not hasattr(np, "object"):


✅ All libraries imported successfully!


2. LOAD MODEL

In [3]:
print("Loading model...")
model = tf.keras.models.load_model('Models/EfficientNetB3/efficientnetb3_model.keras')
model.build((None, 224, 224, 3))
print("✅ Model loaded successfully!")

# Define class names
class_names = [
    'Pepper Bell - Bacterial Spot',
    'Pepper Bell - Healthy',
    'Potato - Early Blight',
    'Potato - Late Blight',
    'Potato - Healthy',
    'Tomato - Target Spot',
    'Tomato - Yellow Leaf Curl Virus',
    'Tomato - Mosaic Virus',
    'Tomato - Healthy'
]

# Disease information
class_info = {
    'Pepper Bell - Bacterial Spot': '🌶️ Bacterial disease causing dark spots on leaves and fruits.',
    'Pepper Bell - Healthy': '✅ Your pepper plant looks healthy!',
    'Potato - Early Blight': '🥔 Fungal disease causing dark spots with concentric rings.',
    'Potato - Late Blight': '🥔 Serious fungal disease that can destroy entire crops.',
    'Potato - Healthy': '✅ Your potato plant looks healthy!',
    'Tomato - Target Spot': '🍅 Fungal disease causing circular spots with dark centers.',
    'Tomato - Yellow Leaf Curl Virus': '🍅 Viral disease causing yellowing and curling of leaves.',
    'Tomato - Mosaic Virus': '🍅 Viral disease causing mottled pattern on leaves.',
    'Tomato - Healthy': '✅ Your tomato plant looks healthy!'
}

# Get last conv layer
base_model = model.get_layer('efficientnetb3')
last_conv_layer_name = None
for layer in reversed(base_model.layers):
    if 'conv' in layer.name.lower():
        last_conv_layer_name = layer.name
        break

print(f"🎯 Using layer: {last_conv_layer_name}")

Loading model...
✅ Model loaded successfully!
🎯 Using layer: top_conv


3. DEFINE GRAD-CAM FUNCTIONS

In [4]:
def make_gradcam_heatmap(img_array, model, last_conv_layer_name):
    """Generate Grad-CAM heatmap"""
    base_model = model.get_layer('efficientnetb3')
    last_conv_layer = base_model.get_layer(last_conv_layer_name)
    
    grad_model = tf.keras.Model(
        inputs=[base_model.input],
        outputs=[last_conv_layer.output, base_model.output]
    )
    
    with tf.GradientTape() as tape:
        last_conv_output, base_predictions = grad_model(img_array)
        
        x = last_conv_output
        for layer in model.layers[1:]:
            x = layer(x)
        predictions = x
        
        pred_index = tf.argmax(predictions[0])
        class_channel = predictions[:, pred_index]
    
    grads = tape.gradient(class_channel, last_conv_output)
    pooled_grads = tf.reduce_mean(grads, axis=(0, 1, 2))
    
    last_conv_output = last_conv_output[0]
    heatmap = last_conv_output @ pooled_grads[..., tf.newaxis]
    heatmap = tf.squeeze(heatmap)
    heatmap = tf.maximum(heatmap, 0) / (tf.math.reduce_max(heatmap) + 1e-10)
    
    return heatmap.numpy()

def overlay_gradcam(img, heatmap, alpha=0.4):
    """Overlay heatmap on image"""
    img = cv2.resize(np.array(img), (224, 224))
    heatmap = cv2.resize(heatmap, (224, 224))
    heatmap = np.uint8(255 * heatmap)
    heatmap = cv2.applyColorMap(heatmap, cv2.COLORMAP_JET)
    superimposed = cv2.addWeighted(img, 1-alpha, heatmap, alpha, 0)
    return superimposed

print("✅ Grad-CAM functions defined!")


✅ Grad-CAM functions defined!
